03 - Exploratory Data Analysis (EDA)

## Setup
Run the cell below first. It detects whether you're in **Google Colab** or
running **locally in VS Code**, and gets the environment ready either way
(clones the repo in Colab, installs requirements, downloads NLTK data, and
adds `src/` to the path so `pipeline.py` can be imported).

In [16]:
import os

repo_root = '/content/NLP_Ctrl-Alt-Elite'

if os.path.exists(repo_root):
    print(f"Contents of '{repo_root}':")
    for root, dirs, files in os.walk(repo_root):
        level = root.replace(repo_root, '').count(os.sep)
        indent = ' ' * 4 * (level)
        print(f'{indent}{os.path.basename(root)}/')
        subindent = ' ' * 4 * (level + 1)
        for f in files:
            print(f'{subindent}{f}')
else:
    print(f"The repository root directory '{repo_root}' does not exist.")

Contents of '/content/NLP_Ctrl-Alt-Elite':
NLP_Ctrl-Alt-Elite/
    src/
        pipeline.py


In [17]:
import os

src_dir = os.path.join(os.getcwd(), 'src')
requirements_path = os.path.join(src_dir, 'requirements.txt')

if os.path.exists(src_dir):
    print(f"Contents of '{src_dir}': {os.listdir(src_dir)}")
    if os.path.exists(requirements_path):
        print(f"'requirements.txt' found in '{src_dir}'.")
    else:
        print(f"'requirements.txt' NOT found in '{src_dir}'.")
else:
    print(f"The directory '{src_dir}' does not exist. Please ensure the repository was cloned correctly.")

Contents of '/content/NLP_Ctrl-Alt-Elite/src': ['pipeline.py']
'requirements.txt' NOT found in '/content/NLP_Ctrl-Alt-Elite/src'.


### Upload `pipeline.py`

Run the cell below to upload your `pipeline.py` file. If the `src` directory does not exist, it will be created.

In [18]:
from google.colab import files
import os

src_dir = '/content/NLP_Ctrl-Alt-Elite/src'

# Create the src directory if it doesn't exist
os.makedirs(src_dir, exist_ok=True)

print(f"Please upload 'pipeline.py' to the directory: {src_dir}")
uploaded = files.upload()

for fname in uploaded:
    if fname == 'pipeline.py':
        destination_path = os.path.join(src_dir, fname)
        with open(destination_path, 'wb') as f:
            f.write(uploaded[fname])
        print(f"Successfully uploaded {fname} to {destination_path}")
    else:
        print(f"Skipping '{fname}'. Please upload 'pipeline.py'.")

print("Upload complete. You can now re-run the preprocessing cell (cell 3c8b0a69).")

Please upload 'pipeline.py' to the directory: /content/NLP_Ctrl-Alt-Elite/src


Saving pipeline.py to pipeline.py
Successfully uploaded pipeline.py to /content/NLP_Ctrl-Alt-Elite/src/pipeline.py
Upload complete. You can now re-run the preprocessing cell (cell 3c8b0a69).


In [19]:
import os, sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL = "https://github.com/Sandaru17513/NLP_Ctrl-Alt-Elite.git"
    REPO_DIR = "NLP_Ctrl-Alt-Elite"

    # Ensure starting in /content for predictable pathing in Colab
    if os.getcwd() != "/content":
        os.chdir("/content")
        print(f"Changed current directory to /content. Current working directory: {os.getcwd()}")

    if not os.path.exists(REPO_DIR):
        print(f"Cloning {REPO_URL} into {os.getcwd()}/{REPO_DIR}")
        get_ipython().system(f"git clone {REPO_URL}")
    else:
        print(f"Repository {REPO_DIR} already exists. Skipping clone.")

    # Now change to the repository root directory
    repo_root_path = os.path.join(os.getcwd(), REPO_DIR)
    if os.path.exists(repo_root_path):
        os.chdir(repo_root_path)
        print(f"Changed working directory to the repository root: {os.getcwd()}")
    else:
        print(f"Error: The repository root directory '{repo_root_path}' does not exist after cloning.")
        print(f"Contents of '{os.getcwd()}': {os.listdir(os.getcwd())}")
        raise FileNotFoundError(f"Repository root directory '{repo_root_path}' not found.")

    get_ipython().system("pip install -q -r requirements.txt")

    # Data files are large - if they were not committed to the repo,
    # upload them here once per Colab session.
    if not os.path.exists("data/data.csv"):
        print("data/data.csv not found in the cloned repo.")
        print("Option A: git add + commit + push the CSVs from your")
        print("          local machine so they come down with the clone.")
        print("Option B: uncomment the lines below to upload manually.")
        # from google.colab import files
        # uploaded = files.upload()   # select data.csv + validation_dataset.csv
        # os.makedirs("data", exist_ok=True)
        # for fname in uploaded:
        #     os.rename(fname, f"data/{fname}")
else:
    print("Running locally (VS Code / Jupyter). Using existing .venv environment.")

import nltk
for pkg in ("punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"):
    try:
        nltk.download(pkg, quiet=True)
    except Exception:
        pass

sys.path.append(os.path.abspath("src")) # 'src' is now relative to the repo root
print("IN_COLAB =", IN_COLAB)
print("Working directory:", os.getcwd())

Changed current directory to /content. Current working directory: /content
Repository NLP_Ctrl-Alt-Elite already exists. Skipping clone.
Changed working directory to the repository root: /content/NLP_Ctrl-Alt-Elite
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'
data/data.csv not found in the cloned repo.
Option A: git add + commit + push the CSVs from your
          local machine so they come down with the clone.
Option B: uncomment the lines below to upload manually.
IN_COLAB = True
Working directory: /content/NLP_Ctrl-Alt-Elite


In [ ]:
get_ipython().system("pip install -q langdetect")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 20.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


### Class distribution

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df = pd.read_csv('../data/dataset.csv')
df_val = pd.read_csv('../data/validation_dataset.csv')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# For df (training data)
# Assuming 'label' column with 0 for ham and 1 for spam
counts = df['label'].value_counts()
ham_count_df = counts.get(0, 0) # Get count for label 0 (ham)
spam_count_df = counts.get(1, 0) # Get count for label 1 (spam)
axes[0].bar(['Ham', 'Spam'], [ham_count_df, spam_count_df],
            color=['#2ecc71', '#e74c3c'], edgecolor='black', linewidth=0.5)
axes[0].set_title('Training Dataset (data.csv)\nClass Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Emails')
for i, v in enumerate([ham_count_df, spam_count_df]):
    axes[0].text(i, v + 200, str(v), ha='center', fontweight='bold')

# For df_val (validation data)
# Assuming 'Email Type' column with 'Safe Email' for ham and 'Phishing Email' for spam
val_counts = df_val['Email Type'].value_counts()
ham_count_val = val_counts.get('Safe Email', 0)
spam_count_val = val_counts.get('Phishing Email', 0)
axes[1].bar(['Ham (Safe)', 'Spam (Phishing)'], [ham_count_val, spam_count_val],
            color=['#3498db', '#e67e22'], edgecolor='black', linewidth=0.5)
axes[1].set_title('Validation Dataset\nClass Distribution', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Number of Emails')
for i, v in enumerate([ham_count_val, spam_count_val]):
    axes[1].text(i, v + 50, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/eda_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to reports/eda_class_distribution.png')

### Text length analysis

In [ ]:
import os
src_dir = '/content/NLP_Ctrl-Alt-Elite/src'

if os.path.exists(src_dir):
    print(f"Contents of '{src_dir}':\n{os.listdir(src_dir)}")
else:
    print(f"The directory '{src_dir}' does not exist.")

Contents of '/content/NLP_Ctrl-Alt-Elite/src':
['pipeline.py']


In [ ]:
import os
print(f"Contents of current directory ({os.getcwd()}):\n{os.listdir('.')}")

Contents of current directory (/content):
['.config', 'NLP_Ctrl-Alt-Elite', 'pipeline.py', 'sample_data']


In [ ]:
import os

repo_root = '/content/NLP_Ctrl-Alt-Elite'

if os.path.exists(repo_root):
    print(f"Contents of '{repo_root}':\n{os.listdir(repo_root)}")
else:
    print(f"The repository root directory '{repo_root}' does not exist. Please ensure the setup cell (4e9af6c3) was run and completed successfully.")

Contents of '/content/NLP_Ctrl-Alt-Elite':
['src']


In [ ]:
import os

src_dir = '../src'
if os.path.exists(src_dir):
    print(f"Files in {src_dir}: {os.listdir(src_dir)}")
else:
    print(f"The directory {src_dir} does not exist.")

In [ ]:
import os
import sys
import pandas as pd

# Ensure ../src is in sys.path, though it should be handled by the setup cell.
src_path = '/content/NLP_Ctrl-Alt-Elite/src/cit-24-01-0182'
if src_path not in sys.path:
    sys.path.append(src_path)

try:
    from pipeline import full_pipeline
except ModuleNotFoundError:
    print("Error: Could not import 'full_pipeline' from 'pipeline.py'.")
    print(f"Please ensure that the file 'pipeline.py' exists in the directory: {src_path}")
    print("And that it contains the 'full_pipeline' function.")
    raise
except ImportError as e:
    print(f"An error occurred while importing from pipeline.py: {e}")
    print(f"Please ensure that the 'full_pipeline' function is defined in 'pipeline.py'.")
    raise

def preprocess_and_save(input_csv_path, output_csv_path, text_col, label_col=None, map_labels=False):
    """
    Reads a CSV, applies full_pipeline to the specified text column,
    renames columns, and saves the preprocessed data.
    """
    print(f"Loading data from {input_csv_path}...")
    df = pd.read_csv(input_csv_path)

    print(f"Applying full_pipeline to column '{text_col}'...")
    df['normalized_text'] = df[text_col].apply(full_pipeline)

    # Rename label column if specified and exists
    if label_col and label_col in df.columns:
        df.rename(columns={label_col: 'email_label'}, inplace=True)
        if map_labels:
            # Map labels to 'ham' and 'spam' strings for consistency with EDA
            if df['email_label'].dtype == 'object': # e.g., 'Safe Email', 'Phishing Email'
                df['email_label'] = df['email_label'].replace({'Safe Email': 'ham', 'Phishing Email': 'spam'})
            elif df['email_label'].dtype == 'int64': # e.g., 0, 1
                df['email_label'] = df['email_label'].replace({0: 'ham', 1: 'spam'})


    # Select and reorder columns as expected by EDA
    if 'email_label' in df.columns:
        df_processed = df[['email_label', 'normalized_text']]
    else:
        df_processed = df[['normalized_text']] # If no label column was processed

    print(f"Saving preprocessed data to {output_csv_path}...")
    df_processed.to_csv(output_csv_path, index=False)
    print(f"Successfully saved {len(df_processed)} rows.")


# Paths for training data
input_data_path = '../data/dataset.csv'
output_preprocessed_path = '../data/preprocessed.csv'

# Paths for validation data
input_val_data_path = '../data/validation_dataset.csv'
output_val_preprocessed_path = '../data/validation_preprocessed.csv'

# Basic check for input data files
if not os.path.exists(input_data_path):
    print(f"Error: Input training data file not found at {input_data_path}. Please upload 'dataset.csv' to the '../data/' directory.")
elif not os.path.exists(input_val_data_path):
    print(f"Error: Input validation data file not found at {input_val_data_path}. Please upload 'validation_dataset.csv' to the '../data/' directory.")
else:
    print("Starting preprocessing...")
    # Preprocess training data (dataset.csv)
    # Original columns: 'text', 'label'. Desired: 'normalized_text', 'email_label' (mapped 0->ham, 1->spam)
    preprocess_and_save(input_data_path, output_preprocessed_path,
                        text_col='text', label_col='label', map_labels=True)
    print("Training data preprocessing complete.")

    # Preprocess validation data (validation_dataset.csv)
    # Original columns: 'Email Text', 'Email Type'. Desired: 'normalized_text', 'email_label' (mapped 'Safe Email'->ham, 'Phishing Email'->spam)
    preprocess_and_save(input_val_data_path, output_val_preprocessed_path,
                        text_col='Email Text', label_col='Email Type', map_labels=True)
    print("Validation data preprocessing complete.")

    print("Preprocessing finished. You can now try running the EDA cells again.")

In [ ]:
import os

src_path = '/content/NLP_Ctrl-Alt-Elite/src/cit-24-01-0182'
pipeline_file = os.path.join(src_path, 'pipeline.py')

if os.path.exists(pipeline_file):
    print(f"Contents of {pipeline_file}:")
    with open(pipeline_file, 'r') as f:
        print(f.read())
else:
    print(f"Error: pipeline.py not found at {pipeline_file}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the preprocessed DataFrame instead of the original one
df = pd.read_csv('../data/preprocessed.csv')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

df['text_length'] = df['normalized_text'].str.len()
df['word_count'] = df['normalized_text'].str.split().str.len()

for label, color in [('ham', '#2ecc71'), ('spam', '#e74c3c')]:
    subset = df[df['email_label'] == label]['text_length']
    axes[0].hist(subset.clip(upper=3000), bins=50, alpha=0.6, label=label.upper(), color=color)
axes[0].set_title('Text Length Distribution by Class')
axes[0].set_xlabel('Character Count (capped at 3000)')
axes[0].legend()

df.groupby('email_label')['word_count'].mean().plot(kind='bar', ax=axes[1], color=['#2ecc71', '#e74c3c'])
axes[1].set_title('Average Word Count by Class')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('../reports/eda_text_length.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import os

file_path = '../data/preprocessed.csv'
if os.path.exists(file_path):
    print(f"The file '{file_path}' exists.")
else:
    print(f"The file '{file_path}' does NOT exist. Please run the preprocessing cell (cell 3c8b0a69) first.")

### Word frequency analysis & word clouds

In [ ]:
from wordcloud import WordCloud
from collections import Counter
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))

def get_top_words(df_subset, n=20):
    all_text = ' '.join(df_subset['normalized_text'].fillna(''))
    words = [w for w in all_text.split() if w not in stop_words and len(w) > 2]
    return Counter(words).most_common(n)

spam_words = get_top_words(df[df['email_label'] == 'spam'])
ham_words = get_top_words(df[df['email_label'] == 'ham'])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

spam_text = ' '.join(df[df['email_label'] == 'spam']['normalized_text'].fillna(''))
wc_spam = WordCloud(width=600, height=400, background_color='white', colormap='Reds', max_words=50).generate(spam_text)
axes[0].imshow(wc_spam, interpolation='bilinear')
axes[0].axis('off')
axes[0].set_title('SPAM Emails - Top Words', fontsize=14, fontweight='bold', color='red')

ham_text = ' '.join(df[df['email_label'] == 'ham']['normalized_text'].fillna(''))
wc_ham = WordCloud(width=600, height=400, background_color='white', colormap='Greens', max_words=50).generate(ham_text)
axes[1].imshow(wc_ham, interpolation='bilinear')
axes[1].axis('off')
axes[1].set_title('HAM Emails - Top Words', fontsize=14, fontweight='bold', color='green')

plt.tight_layout()
plt.savefig('../reports/eda_wordclouds.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 10 SPAM words:', spam_words[:10])
print('Top 10 HAM words:', ham_words[:10])

### Sentence count analysis (unique to Member 1)

In [ ]:
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
import pandas as pd # Import pandas for pd.Series in calculate_sentence_stats

# Calculate sentence count and average sentence length
def calculate_sentence_stats(text):
    if not isinstance(text, str):
        text = str(text) # Handle non-string inputs like NaN or None

    # Segment into sentences and filter out empty strings
    sentences = [s.strip() for s in sent_tokenize(text) if s.strip()]
    num_sentences = len(sentences)

    if num_sentences == 0:
        return 0, 0.0 # sentence_count, avg_sent_len

    word_counts_per_sentence = []
    for s in sentences:
        words = word_tokenize(s)
        if words: # Only count sentences that produced words
            word_counts_per_sentence.append(len(words))

    if not word_counts_per_sentence: # If no words were found across all sentences
        return num_sentences, 0.0

    total_words = sum(word_counts_per_sentence)
    avg_sentence_length = total_words / len(word_counts_per_sentence) # Average based on sentences that had words
    return num_sentences, avg_sentence_length

# Apply the function to the 'normalized_text' column
df[['sentence_count', 'avg_sent_len']] = df['normalized_text'].apply(
    lambda x: pd.Series(calculate_sentence_stats(x))
)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for label, color in [('ham', '#2ecc71'), ('spam', '#e74c3c')]:
    subset = df[df['email_label'] == label]['sentence_count']
    axes[0].hist(subset.clip(upper=20), bins=20, alpha=0.6, label=label.upper(), color=color)
axes[0].set_title('Sentence Count per Email by Class')
axes[0].set_xlabel('Number of Sentences')
axes[0].legend()

df.groupby('email_label')['avg_sent_len'].mean().plot(kind='bar', ax=axes[1], color=['#2ecc71', '#e74c3c'])
axes[1].set_title('Average Sentence Length (Words) by Class')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('../reports/eda_sentences.png', dpi=150, bbox_inches='tight')
plt.show()

print('Sentence stats by class:')
print(df.groupby('email_label')[['sentence_count', 'avg_sent_len']].mean())

In [ ]:
import os

data_dir = '../data'
if os.path.exists(data_dir):
    print(f"Files in {data_dir}: {os.listdir(data_dir)}")
else:
    print(f"The directory {data_dir} does not exist. Please ensure data files are uploaded or cloned correctly.")

In [ ]:
import os
import sys
import pandas as pd

# Ensure ../src is in sys.path, though it should be handled by the setup cell.
src_path = '/content/NLP_Ctrl-Alt-Elite/src/cit-24-01-0182'
if src_path not in sys.path:
    sys.path.append(src_path)

try:
    from pipeline import full_pipeline
except ModuleNotFoundError:
    print("Error: Could not import 'full_pipeline' from 'pipeline.py'.")
    print(f"Please ensure that the file 'pipeline.py' exists in the directory: {src_path}")
    print("And that it contains the 'full_pipeline' function.")
    raise
except ImportError as e:
    print(f"An error occurred while importing from pipeline.py: {e}")
    print(f"Please ensure that the 'full_pipeline' function is defined in 'pipeline.py'.")
    raise

def preprocess_and_save(input_csv_path, output_csv_path, text_col, label_col=None, map_labels=False):
    """
    Reads a CSV, applies full_pipeline to the specified text column,
    renames columns, and saves the preprocessed data.
    """
    print(f"Loading data from {input_csv_path}...")
    df = pd.read_csv(input_csv_path)

    print(f"Applying full_pipeline to column '{text_col}'...")
    df['normalized_text'] = df[text_col].apply(full_pipeline)

    # Rename label column if specified and exists
    if label_col and label_col in df.columns:
        df.rename(columns={label_col: 'email_label'}, inplace=True)
        if map_labels:
            # Map labels to 'ham' and 'spam' strings for consistency with EDA
            if df['email_label'].dtype == 'object': # e.g., 'Safe Email', 'Phishing Email'
                df['email_label'] = df['email_label'].replace({'Safe Email': 'ham', 'Phishing Email': 'spam'})
            elif df['email_label'].dtype == 'int64': # e.g., 0, 1
                df['email_label'] = df['email_label'].replace({0: 'ham', 1: 'spam'})


    # Select and reorder columns as expected by EDA
    if 'email_label' in df.columns:
        df_processed = df[['email_label', 'normalized_text']]
    else:
        df_processed = df[['normalized_text']] # If no label column was processed

    print(f"Saving preprocessed data to {output_csv_path}...")
    df_processed.to_csv(output_csv_path, index=False)
    print(f"Successfully saved {len(df_processed)} rows.")


# Paths for training data
input_data_path = '../data/dataset.csv'
output_preprocessed_path = '../data/preprocessed.csv'

# Paths for validation data
input_val_data_path = '../data/validation_dataset.csv'
output_val_preprocessed_path = '../data/validation_preprocessed.csv'

# Basic check for input data files
if not os.path.exists(input_data_path):
    print(f"Error: Input training data file not found at {input_data_path}. Please upload 'dataset.csv' to the '../data/' directory.")
elif not os.path.exists(input_val_data_path):
    print(f"Error: Input validation data file not found at {input_val_data_path}. Please upload 'validation_dataset.csv' to the '../data/' directory.")
else:
    print("Starting preprocessing...")
    # Preprocess training data (dataset.csv)
    # Original columns: 'text', 'label'. Desired: 'normalized_text', 'email_label' (mapped 0->ham, 1->spam)
    preprocess_and_save(input_data_path, output_preprocessed_path,
                        text_col='text', label_col='label', map_labels=True)
    print("Training data preprocessing complete.")

    # Preprocess validation data (validation_dataset.csv)
    # Original columns: 'Email Text', 'Email Type'. Desired: 'normalized_text', 'email_label' (mapped 'Safe Email'->ham, 'Phishing Email'->spam)
    preprocess_and_save(input_val_data_path, output_val_preprocessed_path,
                        text_col='Email Text', label_col='Email Type', map_labels=True)
    print("Validation data preprocessing complete.")

    print("Preprocessing finished. You can now try running the EDA cells again.")

In [ ]:
import os

src_dir = '/content/NLP_Ctrl-Alt-Elite/src'
if os.path.exists(src_dir):
    print(f"Files in {src_dir}: {os.listdir(src_dir)}")
else:
    print(f"The directory {src_dir} does not exist.")

In [ ]:
import os

src_dir = '/content/NLP_Ctrl-Alt-Elite/src'

if os.path.exists(src_dir):
    print(f"Files in {src_dir}: {os.listdir(src_dir)}")
else:
    print(f"The directory {src_dir} does not exist.")